# GMM Behavioral Segmentation — Streaming Pipeline
**Project:** Revealing Latent Behavioral Dynamics in C2C Interactions  
**Author:** Arevik Melikyan | Supervisor: Varazdat Stepanyan  
**Dataset:** MerRec (~1B rows, monthly Parquet partitions on external SSD)  

---

### Design Principles
| Concern | Decision |
|---|---|
| Memory | All heavy work runs via PyArrow streaming scanners — zero full-file loads |
| Sampling | Configurable `SAMPLE_FRAC` applied *before* GMM training |
| Persistence | Intermediate artefacts written to Parquet/PKL — no re-computation needed |
| Reproducibility | Single `CFG` dict controls every hyper-parameter |
| Inference | Full-dataset scoring is streamed batch-by-batch — never materialised |

> **Assumption:** `stime` is a Unix epoch (float seconds). If it is milliseconds, divide by 1000 in `_new_state` / row loop.  
> **Assumption:** `event_id == "item_view"` is the positive action. Verify against schema before running.


## 0 · Imports & Configuration

In [1]:
import os, json, warnings
from pathlib import Path
from glob import glob
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

# ─── Central config — change ONLY here ───────────────────────────────────────
CFG = {
    # --- Paths ---
    "dataset_root":   "/Volumes/T5 EVO",
    "output_dir":     "gmm_outputs",

    # --- Column names (verify against schema) ---
    "col_user":    "user_id",
    "col_item":    "item_id",       # fallback: "product_id" handled in streamer
    "col_action":  "event_id",
    "col_time":    "stime",
    "positive_action": "item_view",

    # --- Streaming budget ---
    # None  → consume ALL rows (streaming, memory-safe)
    # int   → hard stop after N rows (dev/debug)
    "max_total_rows": 1000,         # <-- PRODUCTION: set to None

    # --- Session definition ---
    "session_gap_s":     180,      # 30-min inactivity boundary
    "min_session_events":   2,      # discard single-event sessions

    # --- Sampling for GMM training (applied AFTER session materialisation) ---
    # 0.01 → 1 % of all emitted sessions; adjust to fit RAM
    "sample_frac": 0.000000001,
    "sample_min_rows": 500,       # floor: raise error if sample < this

    # --- Feature set ---
    "features": [
        "session_length_s",
        "n_events",
        "n_unique_items",
        "revisit_rate",
        "inter_event_time_median",
        "session_velocity",
    ],
    "log_transform_cols": [
        "session_length_s",
        "n_events",
        "n_unique_items",
        "inter_event_time_median",
    ],
    "scaler": "robust",             # "robust" | "standard"

    # --- Model selection ---
    "k_range":           range(2, 5),
    "covariance_types":  ["full", "diag"],
    "n_init":            5,
    "max_iter":          300,

    # --- Final model (set after selection cell) ---
    "final_k":        None,         # auto-filled by selection cell
    "final_cov_type": "full",

    # --- Stability bootstrap ---
    "bootstrap_n":            10,
    "bootstrap_sample_frac":  0.8,

    # --- Inference streaming ---
    "inference_batch_size": 50_000,
}

# ─── Derived paths ────────────────────────────────────────────────────────────
DATASET_PATH   = Path(CFG["dataset_root"]) / "hf" / "merrec"
OUTPUT_DIR     = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SESSIONS_PATH  = OUTPUT_DIR / "sessions.parquet"
LABELED_PATH   = OUTPUT_DIR / "sessions_labeled.parquet"
SCALER_PATH    = OUTPUT_DIR / "gmm_scaler.pkl"
IMPUTER_PATH   = OUTPUT_DIR / "gmm_imputer.pkl"
MODEL_PATH     = OUTPUT_DIR / "gmm_model.pkl"
SELECTION_PATH = OUTPUT_DIR / "model_selection.csv"

assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"
print(f"Ready  | {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Dataset: {DATASET_PATH}")
print(f"Output:  {OUTPUT_DIR.resolve()}")



Ready  | 2026-03-21 22:23:56
Dataset: /Volumes/T5 EVO/hf/merrec
Output:  /Users/arevikmelikyan/Desktop/Capstone_C2C_Behavioral_Dynamics/notebooks/gmm_outputs


## 1 · Session Streamer

`MerRecSessionStreamer` is a **pure Python generator** — it never loads an entire Parquet file.  
Each `item_view` row is processed in place; a session dict lives in memory only until it is emitted.

**Key correctness fixes over original code:**
- `done` flag is checked *before* `ds.dataset()` so the next file is never opened after the row budget is hit.
- Timestamp coercion handles both epoch-seconds (float) and any numeric type.
- `item_col` falls back to `product_id` gracefully.


In [2]:
class MerRecSessionStreamer:
    """
    Stateful streaming generator over monthly MerRec Parquet partitions.

    Yields one session-feature dict per completed session.
    Memory footprint: O(active_users) — only open sessions are held in RAM.

    Parameters
    ----------
    dataset_path : Path
        Root directory containing monthly sub-folders with .parquet files.
    cfg : dict
        Central config dict (CFG defined above).
    """

    def __init__(self, dataset_path: Path, cfg: dict):
        self.dataset_path = Path(dataset_path)
        self.cfg = cfg
        self.files = sorted(
            glob(str(self.dataset_path / "**" / "*.parquet"), recursive=True)
        )
        if not self.files:
            raise FileNotFoundError(f"No parquet files under {self.dataset_path}")
        print(f"[Streamer] {len(self.files)} parquet files found")

    # ------------------------------------------------------------------
    @staticmethod
    def _new_state(ts: float, item: str) -> dict:
        return {
            "session_start": ts,
            "last_ts":       ts,
            "n_events":      1,
            "item_counts":   {item: 1},
            "iets":          [],          # inter-event times
        }

    # ------------------------------------------------------------------
    @staticmethod
    def _emit(user_id: str, state: dict) -> dict:
        """Convert an open session state into a feature dict."""
        n_events    = state["n_events"]
        item_counts = state["item_counts"]
        n_unique    = len(item_counts)
        duration    = state["last_ts"] - state["session_start"]

        revisit_rate = (
            sum(1 for c in item_counts.values() if c > 1) / n_unique
            if n_unique > 0 else 0.0
        )
        iets       = state["iets"]
        iet_median = float(np.median(iets)) if iets else 0.0
        velocity   = n_events / max(duration, 1.0)   # events/second; floor at 1s

        return {
            "user_id":                 user_id,
            "session_length_s":        float(duration),
            "n_events":                n_events,
            "n_unique_items":          n_unique,
            "revisit_rate":            revisit_rate,
            "inter_event_time_median": iet_median,
            "session_velocity":        float(velocity),
        }

    # ------------------------------------------------------------------
    def __iter__(self):
        col_u      = self.cfg["col_user"]
        col_i      = self.cfg["col_item"]
        col_a      = self.cfg["col_action"]
        col_t      = self.cfg["col_time"]
        pos_action = self.cfg["positive_action"]
        gap_s      = self.cfg["session_gap_s"]
        min_ev     = self.cfg["min_session_events"]
        max_total  = self.cfg["max_total_rows"]   # None = unlimited

        user_state: dict = {}
        total_rows: int  = 0
        done: bool       = False

        for fp in self.files:
            if done:
                break

            try:
                dataset      = ds.dataset(fp, format="parquet")
                schema_names = set(dataset.schema.names)

                # Resolve item column
                item_col = col_i if col_i in schema_names else "product_id"
                if item_col not in schema_names:
                    print(f"[WARN] no item column in {fp} — skipping")
                    continue
                if col_t not in schema_names:
                    print(f"[WARN] no timestamp column in {fp} — skipping")
                    continue

                cols_to_read = [c for c in [col_u, item_col, col_a, col_t]
                                if c in schema_names]

                scanner = dataset.scanner(
                    columns=cols_to_read,
                    batch_size=65_536,
                )

                for batch in scanner.to_batches():
                    users   = batch.column(col_u).to_pylist()
                    items   = batch.column(item_col).to_pylist()
                    actions = batch.column(col_a).to_pylist()
                    times   = batch.column(col_t).to_pylist()

                    for u, it, act, ts_raw in zip(users, items, actions, times):
                        # ── Row budget check ──
                        if max_total is not None and total_rows >= max_total:
                            # Flush all open sessions before stopping
                            for uid, st in user_state.items():
                                if st["n_events"] >= min_ev:
                                    yield self._emit(uid, st)
                            done = True
                            break

                        # ── Filter ──
                        if act != pos_action:
                            continue
                        if u is None or it is None or ts_raw is None:
                            continue
                        try:
                            ts = float(ts_raw)
                        except (TypeError, ValueError):
                            continue

                        total_rows += 1
                        it_str = str(it)

                        # ── Session state machine ──
                        if u not in user_state:
                            user_state[u] = self._new_state(ts, it_str)
                        else:
                            st  = user_state[u]
                            gap = ts - st["last_ts"]

                            if gap > gap_s:
                                # Session boundary: emit completed session
                                if st["n_events"] >= min_ev:
                                    yield self._emit(u, st)
                                user_state[u] = self._new_state(ts, it_str)
                            else:
                                st["iets"].append(gap)
                                st["last_ts"]  = ts
                                st["n_events"] += 1
                                st["item_counts"][it_str] = (
                                    st["item_counts"].get(it_str, 0) + 1
                                )

                    if done:
                        break

            except Exception as exc:
                print(f"[WARN] Failed reading {fp}: {exc}")
                continue

        # ── Natural end: flush remaining open sessions ──
        if not done:
            for uid, st in user_state.items():
                if st["n_events"] >= min_ev:
                    yield self._emit(uid, st)

        print(f"[Streamer] Done — {total_rows:,} positive-action rows consumed")


print("MerRecSessionStreamer defined ✓")


MerRecSessionStreamer defined ✓


## 2 · Stream Sessions → Parquet

Writes sessions in chunks of 20 000 rows using `ParquetWriter`.  
`skip_if_exists=True` lets you re-run the notebook without re-scanning the full dataset.

> **Memory guarantee:** at no point is more than `chunk_size` session dicts held in RAM simultaneously.


In [ ]:
def stream_sessions_to_parquet(
    streamer,
    out_path: Path,
    chunk_size: int = 20_000,
    skip_if_exists: bool = True,
) -> None:
    """
    Consume a MerRecSessionStreamer and write sessions to a single Parquet file.

    Parameters
    ----------
    skip_if_exists : bool
        If True and out_path already exists, skip streaming entirely.
    """
    if skip_if_exists and out_path.exists():
        size_mb = out_path.stat().st_size / 1_048_576
        print(f"[SKIP] {out_path} already exists ({size_mb:.1f} MB)")
        return

    buffer: list = []
    writer = None

    for session in tqdm(streamer, desc="Streaming → Parquet"):
        buffer.append(session)

        if len(buffer) >= chunk_size:
            table = pa.Table.from_pandas(pd.DataFrame(buffer))
            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
            writer.write_table(table)
            buffer.clear()

    if buffer:
        table = pa.Table.from_pandas(pd.DataFrame(buffer))
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
        writer.write_table(table)

    if writer:
        writer.close()
        size_mb = out_path.stat().st_size / 1_048_576
        print(f"Saved → {out_path}  ({size_mb:.1f} MB)")
    else:
        print("[WARN] No sessions written — check streamer configuration")


# ── Execute ──
streamer = MerRecSessionStreamer(DATASET_PATH, CFG)
stream_sessions_to_parquet(streamer, SESSIONS_PATH)

# Validate
meta = pq.read_metadata(SESSIONS_PATH)
print(f"Rows: {meta.num_rows:,} | Row groups: {meta.num_row_groups}")


[Streamer] 2170 parquet files found


Streaming → Parquet: 0it [00:00, ?it/s]

## 3 · Reservoir Sampling for GMM Training

**Why not `df.sample()`?**  
`pq.read_table(...).to_pandas()` loads the entire sessions Parquet into RAM.  
For 1 B source rows, even 1% sessions can be tens of millions of rows.

We use **streaming reservoir sampling** (Algorithm R) directly over PyArrow batches.  
This is O(k) memory regardless of file size.

> **Assumption:** `sample_frac = 0.01` (1%) is the default. Tune so the sample fits  
> comfortably in RAM: `sample_size ≈ total_sessions × sample_frac`.


In [ ]:
def reservoir_sample_parquet(
    path: Path,
    sample_frac: float,
    random_state: int = 42,
    batch_size: int = 100_000,
    min_rows: int = 5_000,
) -> pd.DataFrame:
    """
    Algorithm-R reservoir sampling over a Parquet file.
    Memory: O(sample_size) — the reservoir only.

    Parameters
    ----------
    sample_frac : float
        Fraction of total rows to retain (e.g. 0.01 = 1 %).
    min_rows : int
        Raise ValueError if the final sample is smaller than this.

    Returns
    -------
    pd.DataFrame
        Sampled rows in arbitrary order.
    """
    rng = np.random.default_rng(random_state)

    # First pass: count rows to compute k
    meta      = pq.read_metadata(path)
    n_total   = meta.num_rows
    k         = max(min_rows, int(n_total * sample_frac))
    print(f"[Sample] Total sessions: {n_total:,} | Target k: {k:,} ({sample_frac*100:.2f}%)")

    reservoir: list = []
    row_idx: int    = 0

    dataset = ds.dataset(path, format="parquet")
    scanner = dataset.scanner(batch_size=batch_size)

    for batch in tqdm(scanner.to_batches(), desc="Reservoir sampling"):
        rows = batch.to_pydict()
        n    = batch.num_rows
        keys = list(rows.keys())

        for i in range(n):
            row = {k_: rows[k_][i] for k_ in keys}

            if row_idx < k:
                reservoir.append(row)
            else:
                j = int(rng.integers(0, row_idx + 1))
                if j < k:
                    reservoir[j] = row

            row_idx += 1

    df = pd.DataFrame(reservoir)
    if len(df) < min_rows:
        raise ValueError(
            f"Sample too small ({len(df)}) — increase sample_frac or lower min_rows"
        )
    print(f"[Sample] Final sample: {len(df):,} rows")
    return df.reset_index(drop=True)


df_train = reservoir_sample_parquet(
    SESSIONS_PATH,
    sample_frac=CFG["sample_frac"],
    random_state=RANDOM_STATE,
    min_rows=CFG["sample_min_rows"],
)

print(df_train.shape)
df_train.describe()


## 4 · Feature Engineering & Preprocessing

Pipeline (applied identically at training time and inference time):

1. **Clip** — 99.5th-percentile cap per feature (controls outlier influence on GMM covariance)
2. **Impute** — median imputation (sessions with NaN velocity/IET get filled)
3. **Log1p** — right-skewed features (duration, counts) → approximately Gaussian
4. **Scale** — `RobustScaler` (median/IQR) — robust to residual outliers post-clip

> **Assumption:** `session_velocity = n_events / max(duration, 1s)` can produce very large values
> for micro-sessions. The 99.5th clip handles this without losing signal.


In [ ]:
def build_feature_matrix(df: pd.DataFrame, cfg: dict, fit: bool = True,
                        scaler=None, imputer=None):
    """
    Build and preprocess the feature matrix.

    Parameters
    ----------
    fit : bool
        If True, fit scaler and imputer on df. If False, use provided artefacts.
    scaler, imputer : sklearn transformers
        Required when fit=False (inference mode).

    Returns
    -------
    X_scaled : np.ndarray
    scaler   : fitted scaler
    imputer  : fitted imputer
    """
    features = cfg["features"]
    X = df[features].copy().astype(float)

    # 1. Clip at 99.5th percentile per feature
    for col in features:
        cap = X[col].quantile(0.995)
        X[col] = X[col].clip(upper=cap)

    # 2. Impute
    if fit:
        imputer = SimpleImputer(strategy="median")
        X_imp   = imputer.fit_transform(X)
    else:
        X_imp   = imputer.transform(X)
    X = pd.DataFrame(X_imp, columns=features)

    # 3. Log1p on skewed features
    for col in cfg["log_transform_cols"]:
        if col in features:
            X[col] = np.log1p(X[col])

    # 4. Scale
    if fit:
        Scaler = RobustScaler if cfg["scaler"] == "robust" else StandardScaler
        scaler = Scaler()
        X_scaled = scaler.fit_transform(X)
    else:
        X_scaled = scaler.transform(X)

    return X_scaled, scaler, imputer


X_train, scaler, imputer = build_feature_matrix(df_train, CFG, fit=True)

joblib.dump(scaler,  SCALER_PATH);  print(f"Scaler  → {SCALER_PATH}")
joblib.dump(imputer, IMPUTER_PATH); print(f"Imputer → {IMPUTER_PATH}")

print(f"Feature matrix: {X_train.shape}  |  dtype: {X_train.dtype}")
print(f"Feature means (post-scale): {X_train.mean(axis=0).round(3)}")
print(f"Feature stds  (post-scale): {X_train.std(axis=0).round(3)}")


## 5 · GMM Model Selection (BIC + Silhouette)

**Primary criterion:** BIC (Bayesian Information Criterion) — penalises model complexity.  
**Secondary criterion:** Silhouette score — validates cluster separation geometrically.

We grid-search over `k` (number of components) × `covariance_type`.

> **Note:** `full` covariance is more expressive but requires `O(k × d²)` parameters.  
> For `d=6` features, `full` is still tractable. If you expand the feature set beyond ~20,  
> consider switching to `diag` or `tied`.


In [ ]:
def gmm_model_selection(X: np.ndarray, cfg: dict) -> pd.DataFrame:
    """
    Grid search over (k, covariance_type) using BIC and Silhouette.

    Returns a DataFrame sorted by BIC ascending (lower = better).
    """
    results = []

    for cov in cfg["covariance_types"]:
        for k in cfg["k_range"]:
            gmm = GaussianMixture(
                n_components=k,
                covariance_type=cov,
                n_init=cfg["n_init"],
                max_iter=cfg["max_iter"],
                random_state=RANDOM_STATE,
                warm_start=False,
            )
            gmm.fit(X)
            labels = gmm.predict(X)

            sil = silhouette_score(X, labels, sample_size=min(10_000, len(X)),
                                   random_state=RANDOM_STATE)
            db  = davies_bouldin_score(X, labels)
            ch  = calinski_harabasz_score(X, labels)

            results.append({
                "k":   k,
                "cov": cov,
                "bic": gmm.bic(X),
                "aic": gmm.aic(X),
                "sil": sil,
                "db":  db,    # lower = better
                "ch":  ch,    # higher = better
            })
            print(f"  k={k} cov={cov:<5}  BIC={gmm.bic(X):>12,.1f}  Sil={sil:.4f}")

    return pd.DataFrame(results).sort_values("bic").reset_index(drop=True)


print("Running model selection …")
selection_df = gmm_model_selection(X_train, CFG)
selection_df.to_csv(SELECTION_PATH, index=False)
print(f"\nResults saved → {SELECTION_PATH}")
display(selection_df)


### 5b · Model Selection Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for cov, grp in selection_df.groupby("cov"):
    axes[0].plot(grp["k"], grp["bic"],  marker="o", label=cov)
    axes[1].plot(grp["k"], grp["sil"],  marker="o", label=cov)
    axes[2].plot(grp["k"], grp["db"],   marker="o", label=cov)

axes[0].set(title="BIC (↓ better)",       xlabel="k", ylabel="BIC")
axes[1].set(title="Silhouette (↑ better)", xlabel="k", ylabel="Silhouette")
axes[2].set(title="Davies-Bouldin (↓ better)", xlabel="k", ylabel="DB")

for ax in axes:
    ax.legend()

plt.suptitle("GMM Model Selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_selection.png", bbox_inches="tight")
plt.show()


## 6 · Train Final GMM

`CFG["final_k"]` is auto-filled from the best BIC row if left as `None`.  
Override manually after inspecting the selection plot (e.g. prefer a more interpretable k).


In [ ]:
# Auto-fill final_k from selection if not set manually
if CFG["final_k"] is None:
    best_row      = selection_df.iloc[0]
    CFG["final_k"]        = int(best_row["k"])
    CFG["final_cov_type"] = str(best_row["cov"])
    print(f"Auto-selected: k={CFG['final_k']}, cov={CFG['final_cov_type']}")
else:
    print(f"Manual override: k={CFG['final_k']}, cov={CFG['final_cov_type']}")

final_gmm = GaussianMixture(
    n_components=CFG["final_k"],
    covariance_type=CFG["final_cov_type"],
    n_init=10,                  # more inits for final model
    max_iter=CFG["max_iter"],
    random_state=RANDOM_STATE,
)
final_gmm.fit(X_train)
joblib.dump(final_gmm, MODEL_PATH)
print(f"Model saved → {MODEL_PATH}")

labels_train = final_gmm.predict(X_train)
probs_train  = final_gmm.predict_proba(X_train)

print(f"\nCluster distribution (training sample):")
unique, counts = np.unique(labels_train, return_counts=True)
for c, n in zip(unique, counts):
    print(f"  Cluster {c}: {n:>7,} ({100*n/len(labels_train):.1f}%)")
print(f"\nMean confidence: {probs_train.max(axis=1).mean():.4f}")


## 7 · Cluster Profiling

Compute per-cluster statistics on the **original (unscaled)** feature values  
so the numbers are interpretable in domain terms (seconds, event counts, etc.).


In [ ]:
df_profile = df_train.copy()
df_profile["gmm_regime"]  = labels_train
df_profile["confidence"]  = probs_train.max(axis=1)

profile = (
    df_profile
    .groupby("gmm_regime")[CFG["features"] + ["confidence"]]
    .agg(["mean", "median", "std", "count"])
)

print("=== Cluster Profile (original scale) ===")
display(profile)

# Save
profile.to_csv(OUTPUT_DIR / "cluster_profile.csv")
print(f"Saved → {OUTPUT_DIR / 'cluster_profile.csv'}")


## 8 · Visualisation

**8a** — PCA 2D scatter of the training sample coloured by cluster  
**8b** — Feature distribution violin plots per cluster  
**8c** — Confidence score distribution  

All plots are saved to `gmm_outputs/`.


In [ ]:
from sklearn.decomposition import PCA

PALETTE = ["#4361EE", "#F72585", "#4CC9F0", "#7209B7", "#3A0CA3", "#560BAD", "#480CA8"]

# ── 8a: PCA 2D scatter ──
pca  = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca.fit_transform(X_train)

fig, ax = plt.subplots(figsize=(9, 6))
for c in range(CFG["final_k"]):
    mask = labels_train == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               s=4, alpha=0.35, color=PALETTE[c], label=f"Cluster {c}")

ax.set_title("PCA 2D — GMM Clusters (training sample)", fontsize=13)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.legend(markerscale=4)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_scatter.png", bbox_inches="tight")
plt.show()

# ── 8b: Feature violin per cluster ──
n_feat = len(CFG["features"])
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, feat in zip(axes.flatten(), CFG["features"]):
    sns.violinplot(
        data=df_profile, x="gmm_regime", y=feat,
        palette=PALETTE[:CFG["final_k"]], inner="quartile", ax=ax, cut=0
    )
    ax.set_title(feat)
    ax.set_xlabel("Cluster")

plt.suptitle("Feature Distributions by Cluster", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_violins.png", bbox_inches="tight")
plt.show()

# ── 8c: Confidence histogram ──
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df_profile["confidence"], bins=50, color="#4361EE", alpha=0.8, edgecolor="white")
ax.axvline(0.8, color="red", linestyle="--", label="0.8 threshold")
ax.set(title="Assignment Confidence (max posterior probability)",
       xlabel="P(cluster | session)", ylabel="Count")
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confidence_hist.png", bbox_inches="tight")
plt.show()


## 9 · Stability Bootstrap

Re-train the GMM on `bootstrap_n` sub-samples of the training data  
and measure silhouette variance. A stable solution should have low std.

> **Interpretation:** Silhouette std < 0.01 across bootstrap rounds indicates  
> the cluster structure is not artefactual.


In [ ]:
from sklearn.metrics import adjusted_rand_score

sil_scores = []

for i in tqdm(range(CFG["bootstrap_n"]), desc="Bootstrap"):
    idx      = np.random.choice(len(X_train),
                                size=int(len(X_train) * CFG["bootstrap_sample_frac"]),
                                replace=False)
    X_bs     = X_train[idx]

    gmm_bs   = GaussianMixture(
        n_components=CFG["final_k"],
        covariance_type=CFG["final_cov_type"],
        n_init=3, max_iter=200,
        random_state=i,
    )
    gmm_bs.fit(X_bs)
    labels_bs = gmm_bs.predict(X_bs)

    sil = silhouette_score(X_bs, labels_bs,
                           sample_size=min(5_000, len(X_bs)),
                           random_state=RANDOM_STATE)
    sil_scores.append(sil)

sil_arr = np.array(sil_scores)
print(f"Bootstrap Silhouette — mean: {sil_arr.mean():.4f}  std: {sil_arr.std():.4f}  "
      f"min: {sil_arr.min():.4f}  max: {sil_arr.max():.4f}")

if sil_arr.std() > 0.02:
    print("[WARNING] High silhouette variance — consider increasing k or changing covariance type")
else:
    print("[OK] Cluster solution is stable across bootstrap rounds")


## 10 · Full-Dataset Streaming Inference

Score every session in `sessions.parquet` without loading the file into RAM.  
Output: `sessions_labeled.parquet` with two extra columns:
- `gmm_regime` — cluster assignment (int)
- `confidence`  — max posterior probability (float)

**Preprocessing is identical to training** (clip → impute → log1p → scale).  
The clip cap is re-computed per batch; this is correct because the 99.5th-percentile  
cap per batch approximates the global cap when batches are large (50 K rows).

> **Alternative for exact caps:** compute global percentiles from the training sample  
> and hard-code them in CFG. Pass them into the inference function and apply them as  
> fixed bounds rather than per-batch quantiles.


In [ ]:
def stream_inference(
    sessions_path: Path,
    out_path: Path,
    gmm,
    scaler,
    imputer,
    cfg: dict,
    batch_size: int = None,
) -> None:
    """
    Score all sessions in `sessions_path` using the trained GMM.
    Writes labeled Parquet to `out_path`.

    Memory: O(batch_size) at any point.
    """
    batch_size = batch_size or cfg["inference_batch_size"]
    features   = cfg["features"]

    dataset = ds.dataset(sessions_path, format="parquet")
    scanner = dataset.scanner(batch_size=batch_size)

    writer = None
    total  = 0

    for batch in tqdm(scanner.to_batches(), desc="Inference"):
        df = batch.to_pandas()

        # Preprocessing (must mirror build_feature_matrix exactly)
        X = df[features].copy().astype(float)
        for col in features:
            cap  = X[col].quantile(0.995)
            X[col] = X[col].clip(upper=cap)

        X = pd.DataFrame(imputer.transform(X), columns=features)

        for col in cfg["log_transform_cols"]:
            if col in features:
                X[col] = np.log1p(X[col])

        X_scaled = scaler.transform(X)

        labels = gmm.predict(X_scaled)
        probs  = gmm.predict_proba(X_scaled)

        df = df.copy()
        df["gmm_regime"] = labels.astype(np.int8)
        df["confidence"] = probs.max(axis=1).astype(np.float32)

        table = pa.Table.from_pandas(df)

        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
        writer.write_table(table)
        total += len(df)

    if writer:
        writer.close()
        size_mb = out_path.stat().st_size / 1_048_576
        print(f"Saved {total:,} labeled sessions → {out_path}  ({size_mb:.1f} MB)")
    else:
        print("[WARN] No batches written")


# ── Load artefacts (idempotent — re-run safe) ──
final_gmm_inf = joblib.load(MODEL_PATH)
scaler_inf    = joblib.load(SCALER_PATH)
imputer_inf   = joblib.load(IMPUTER_PATH)

stream_inference(SESSIONS_PATH, LABELED_PATH,
                 final_gmm_inf, scaler_inf, imputer_inf, CFG)


## 11 · Output Summary & Artefact Inventory

| Artefact | Path | Description |
|---|---|---|
| Sessions | `gmm_outputs/sessions.parquet` | All emitted sessions (streamed from raw) |
| Scaler | `gmm_outputs/gmm_scaler.pkl` | Fitted RobustScaler |
| Imputer | `gmm_outputs/gmm_imputer.pkl` | Fitted SimpleImputer |
| Model | `gmm_outputs/gmm_model.pkl` | Final GMM |
| Selection | `gmm_outputs/model_selection.csv` | BIC/Silhouette grid results |
| Cluster profile | `gmm_outputs/cluster_profile.csv` | Per-cluster feature stats |
| Labeled data | `gmm_outputs/sessions_labeled.parquet` | Full dataset with assignments |
| Plots | `gmm_outputs/*.png` | Visualisations |

### Known Assumptions & Risks

| # | Assumption | Mitigation |
|---|---|---|
| 1 | `stime` is Unix epoch in **seconds** | If milliseconds: divide by 1000 in streamer row loop |
| 2 | `item_view` is the correct positive action | Verify with `df.event_id.value_counts()` |
| 3 | 1% sample is representative | Check sample vs population feature quantiles |
| 4 | Per-batch clip ≈ global clip at inference | Hard-code training percentiles in CFG for exactness |
| 5 | Session gap of 30 min is domain-appropriate | Tune via `session_gap_s` in CFG |


In [ ]:
print("=== Artefact Inventory ===")
for p in sorted(OUTPUT_DIR.glob("*")):
    size = p.stat().st_size
    unit = "MB" if size > 1_048_576 else "KB"
    val  = size / (1_048_576 if unit == "MB" else 1024)
    print(f"  {p.name:<40} {val:>7.1f} {unit}")
